# Archetypal SAEs ablations

In this notebook, we show the fatal flow of the archetypal SAEs https://arxiv.org/pdf/2502.12892
- The authors claim that the archetypal SAEs are more stable than standard ones, where stability measure how closes are dictionaries trained with a different seed
- However, we suspect that their setting is unfair: they always initialize archetypal SAEs the same while the standard SAEs are initialized at random
- We check that the stability comes solely from the initialization advantage and not al all from archetypal analysis

We test this conjecture by performing the following ablation studies:
- We check that the archetypal SAEs perform the same with archetypal projection switched off
- We check that the standard topK SAEs achieve the same good stability metric when we always initialize them at the k-means centroids and turn off normalization

For fair test, this notebook uses the code and the minimal example provided by the authors of the paper: https://github.com/KempnerInstitute/overcomplete

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# importing the minimal set of lib
# and download the small rabbit dataset

import torch
import numpy as np
import matplotlib.pyplot as plt

from einops import rearrange
from torchvision import transforms

import overcomplete
from overcomplete.metrics import cosine_hungarian_loss



## Load the dataset

In [ ]:
# Uncomment for loading the dataset
# !wget -O rabbit.npz -q "https://github.com/KempnerInstitute/Overcomplete/blob/main/docs/assets/rabbit.npz?raw=True"

In [ ]:
from overcomplete.visualization import show
# show is able to plot any numpy/pil/tensor and handle various
# data format (channel first, last, no channel...)

original_images = np.load('rabbit.npz')['arr_0'].astype(np.uint8)
for i in range(10):
  plt.subplot(2, 5, i+1)
  show(original_images[i])
plt.show()

## Load the DinoV2 vision encoder and get patch emebeddings

In [ ]:
images = torch.Tensor(original_images).cuda()
images.shape
from overcomplete.models import DinoV2
model = DinoV2(device='cuda')

images = model.processor(images=images, return_tensors="pt").to("cuda")
last_hidden_states = model.forward_features(images)
patch_tokens = last_hidden_states[:,1:,:]

Activations = rearrange(patch_tokens, 'n t d -> (n t) d')
Activations.shape

## Train archetypal SAE

### Get centroids from k-means

In [ ]:
# before training our SAE, we need to capture "points" that will define the convex hull
# you can think of them as landmark that will guide the dictionary to be close to
# the activations regions.
# here we use kmeans, be sure to have more centroids than concepts.
# for large scale computation, we recommend to use faiss.
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=100, random_state=0, verbose=1)
kmeans.fit(Activations.cpu())

centroids = kmeans.cluster_centers_
centroids.shape

In [ ]:
# now the funny part, lets train an archetypal topk sae
# we need a dataloader, a criterion and an optimizer
from torch.utils.data import DataLoader, TensorDataset
from overcomplete.sae import RATopKSAE, train_sae

sae = RATopKSAE(Activations.shape[-1], nb_concepts=30, top_k=3, points=torch.tensor(centroids).float().cuda(),
                delta=0.2, device='cuda')

dataloader = torch.utils.data.DataLoader(TensorDataset(Activations), batch_size=1024, shuffle=True)
optimizer = torch.optim.Adam(sae.parameters(), lr=5e-3)

def criterion(x, x_hat, pre_codes, codes, dictionary):
  loss = (x - x_hat).square().mean()

  # add reanimation loss to avoid dead codes:
  is_dead = ((codes > 0).sum(dim=0) == 0).float().detach()
  reanim_loss = (pre_codes * is_dead[None, :]).mean()
  loss -= reanim_loss * 1e-3

  return loss

In [ ]:
logs = train_sae(sae, dataloader, criterion, optimizer, nb_epochs=30, device='cuda')

### Visualize SAE features as heatmaps

In [ ]:
from overcomplete.visualization import overlay_top_heatmaps
# great, now let inspect the results !
# first we reshape the code to see them patch-wise
# then we will use `overlay_top_heatmaps`
# a little function to show the top concepts

sae = sae.eval()

with torch.no_grad():
  pre_codes, codes = sae.encode(Activations)

codes = rearrange(codes, '(n w h) d -> n w h d', w=16, h=16)

for i in range(10):
  print('Concept', i)
  overlay_top_heatmaps(original_images, codes, concept_id=i)
  plt.show()

## Measuring stability

### Train two classical topk saes on different seeds

In [ ]:
# classical topk
from overcomplete.sae import TopKSAE

nb_concepts = 200

def train(sae):
  dataloader = torch.utils.data.DataLoader(TensorDataset(Activations), batch_size=1024, shuffle=True)
  optimizer = torch.optim.Adam(sae.parameters(), lr=5e-3)
  logs = train_sae(sae, dataloader, criterion, optimizer, nb_epochs=20, device='cuda')

sae1 = TopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda')
train(sae1)

sae2 = TopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda')
train(sae2)


### Train two archetypal SAEs on different seeds

In [ ]:
# now with archetypal
# classical topk
rasae1 = RATopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda', points=torch.tensor(centroids).float().cuda(), delta=0.1)
train(rasae1)

rasae2 = RATopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda', points=torch.tensor(centroids).float().cuda(), delta=0.1)
train(rasae2)


In [ ]:
d1, d2 = sae1.get_dictionary(), sae2.get_dictionary()
d3, d4 = rasae1.get_dictionary(), rasae2.get_dictionary()

score_classic_sae = cosine_hungarian_loss(d1.detach(), d2.detach()) / d1.shape[0]
score_archetypal_sae = cosine_hungarian_loss(d3.detach(), d4.detach()) / d1.shape[0]

print(f'Hungarian distance Classic SAE: {score_classic_sae}')
print(f'Hungarian distance Archetypal SAE: {score_archetypal_sae}')

Here we authors claim that the archetypal SAEs are more stable, in the our ablations below we show that their argument is flawedand the stability improvements comes solely from fixing the dictionary initialization. It has nothing to do with their archetypal analysis.

### Turn off archetypal projection in archetypal SAEs

In [ ]:
# now with archetypal
# classical topk
rasae1_ablated = RATopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda', points=torch.tensor(centroids).float().cuda(), delta=0.1, projection=False)
train(rasae1_ablated)

rasae2_ablated = RATopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda', points=torch.tensor(centroids).float().cuda(), delta=0.1, projection=False)
train(rasae2_ablated)

d5, d6 = rasae1_ablated.get_dictionary(), rasae2_ablated.get_dictionary()
score_archetypal_sae_ablated = cosine_hungarian_loss(d5.detach(), d6.detach()) / nb_concepts
print(f'Hungarian distance Archetypal SAE Ablated: {score_archetypal_sae_ablated}')

## Fixing initialization of topk SAEs

In [ ]:
C=torch.tensor(centroids).float().cuda()
nb_concepts = 200
nb_candidates = C.shape[0]
W = torch.eye(nb_concepts, nb_candidates, device="cuda")
D = W @ C
dictionary_params = {"initializer": D, "normalization": "identity"}

In [ ]:
sae1_fixed_init = TopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda', dictionary_params=dictionary_params)
train(sae1_fixed_init)

sae2_fixed_init = TopKSAE(Activations.shape[-1], nb_concepts=nb_concepts, top_k=3, device='cuda', dictionary_params=dictionary_params)
train(sae2_fixed_init)

d7, d8 = sae1_fixed_init.get_dictionary(), sae2_fixed_init.get_dictionary()
score_classic_sae_fixed_init = cosine_hungarian_loss(d7.detach(), d8.detach()) / nb_concepts
print(f'Hungarian distance Classic SAE with fixed init: {score_classic_sae_fixed_init}')

In [ ]:
print(f'Hungarian distance Classic SAE: {score_classic_sae}')
print(f'Hungarian distance Archetypal SAE: {score_archetypal_sae}')
print(f'Hungarian distance Archetypal SAE Ablated: {score_archetypal_sae_ablated}')
print(f'Hungarian distance Classic SAE with fixed init: {score_classic_sae_fixed_init}')